# Model Evaluation Pipeline

This notebook loads the 1st-place models from our production directory, prepares the testing data, and evaluates their performance. 

In [1]:
import os
import sys
import json
import tomllib

import numpy as np
import pandas as pd
import tensorflow as tf
import plotly.graph_objects as go

from datetime import datetime

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

os.chdir("..") 

from scripts.tune_architectures import load_data, create_train_test_data, set_global_determinism

In [2]:
set_global_determinism(seed=42, init_tf_config=True)

## Load Production Models

In [3]:
production_models = {}
prod_dir = "models/production"

print("📥 Loading production models...")

for exp_name in os.listdir(prod_dir):
    exp_path = os.path.join(prod_dir, exp_name)
    
    if os.path.isdir(exp_path):
        model_file = os.path.join(exp_path, "best_model.keras")
        
        if os.path.exists(model_file):
            try:
                production_models[exp_name] = tf.keras.models.load_model(model_file)
                print(f"  ✅ Loaded: {exp_name}")
            except Exception as e:
                print(f"  ❌ Failed to load {exp_name}: {e}")

print(f"\nTotal models loaded: {len(production_models)}")

📥 Loading production models...
  ✅ Loaded: Corn_30
  ✅ Loaded: Corn_7
  ✅ Loaded: Soy_30
  ✅ Loaded: Soy_7
  ✅ Loaded: Wheat_30
  ✅ Loaded: Wheat_7

Total models loaded: 6


## Data Processing and Testing Loop

### Testing prediction for day t + 1

### Single-Step Forecast Evaluation
In this section, we load the agro dataset and iterate through our production models. For each model, we load the corresponding scaler, prepare the test dataset, and predict the immediate next step. The results are compared against the actual prices to calculate RMSE and MAE.


In [4]:
df = load_data('data/trusted/agro_master_table.parquet')

with open("config.toml", "rb") as f:
    config = tomllib.load(f)

experiment_config = config['experiment']
horizon_steps = experiment_config['horizon_steps']
train_pct = experiment_config['train_percentage']
val_pct = experiment_config['validation_percentage']

test_results = []
print("🚀 Starting Evaluation Pipeline...\n")

for exp_name, model in production_models.items():
    print(f"📊 Evaluating Model: {exp_name}")
    
    parts = exp_name.split("_")
    commodity = parts[0].lower()
    time_steps = int(parts[1])
    
    target_col = f"{commodity}_Close" if f"{commodity}_Close" in df.columns else commodity
    all_cols = [target_col]

    scaler_path = f"saved_scalers/{commodity.lower()}/scaler_params.json"

    try:
        with open(scaler_path, "r") as f:
            scaler_params = json.load(f)
            
            scaler = MinMaxScaler(feature_range=tuple(scaler_params['feature_range']))
            
            scaler.min_ = np.array(scaler_params['min_'])
            scaler.scale_ = np.array(scaler_params['scale_'])
            scaler.data_min_ = np.array(scaler_params['data_min_'])
            scaler.data_max_ = np.array(scaler_params['data_max_'])
            
            scaler.data_range_ = scaler.data_max_ - scaler.data_min_
            
            print(f"   ✅ Scaler loaded successfully for {commodity}")
    except Exception as e:
        print(f"   ❌ Failed to load scaler at {scaler_path}: {e}")
        continue 
    
    X_train, y_train, X_val_rec, y_val_rec, X_test, y_test = create_train_test_data(
        df, all_cols, time_steps, 
        horizon_steps=horizon_steps, 
        train_pct=train_pct, 
        val_pct=val_pct
    )
    
    predictions = model.predict(X_test, verbose=0)
    
    predictions_real = scaler.inverse_transform(predictions.reshape(-1, 1)).flatten()
    y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    
    rmse = np.sqrt(mean_squared_error(y_test_real, predictions_real))
    mae = mean_absolute_error(y_test_real, predictions_real)
    
    # =================================
    # === Visualization with Plotly ===
    # =================================

    
    train_size = int(len(df) * train_pct)
    val_size = int(len(df) * val_pct)
    
    test_dates = df['Date'].iloc[train_size + val_size + time_steps : ]

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=test_dates,
        y=y_test_real, 
        mode='lines', 
        name='Actual Price',
        line=dict(color='#1f77b4', width=2)
    ))

    # Add Predicted Data Trace
    fig.add_trace(go.Scatter(
        x=test_dates,
        y=predictions_real, 
        mode='lines', 
        name='Model Forecast',
        line=dict(color='#ff7f0e', width=2, dash='dash') 
    ))

    current_date = datetime.now().strftime("%B %Y")

    fig.update_layout(
        title=f"<b>{exp_name}</b>: Forecast vs Actual<br><sup>Evaluation Run: {current_date}</sup>",
        xaxis_title="Date",
        yaxis_title="Price",
        template="plotly_white",
        hovermode="x unified",
        margin=dict(l=40, r=40, t=80, b=40),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    fig.show()

    test_results.append({
        "Model": exp_name,
        "Commodity": commodity,
        "Time Steps": time_steps,
        "Test RMSE": rmse,
        "Test MAE": mae
    })

results_df = pd.DataFrame(test_results)
print("\n📊 Final Evaluation Results:")
display(results_df)

🚀 Starting Evaluation Pipeline...

📊 Evaluating Model: Corn_30
   ✅ Scaler loaded successfully for corn


📊 Evaluating Model: Corn_7
   ✅ Scaler loaded successfully for corn


📊 Evaluating Model: Soy_30
   ✅ Scaler loaded successfully for soy


📊 Evaluating Model: Soy_7
   ✅ Scaler loaded successfully for soy


📊 Evaluating Model: Wheat_30
   ✅ Scaler loaded successfully for wheat


📊 Evaluating Model: Wheat_7
   ✅ Scaler loaded successfully for wheat



📊 Final Evaluation Results:


,Model,Commodity,Time Steps,Test RMSE,Test MAE
0,Corn_30,corn,30,9.636086,7.462903
1,Corn_7,corn,7,6.693390,5.043814
2,Soy_30,soy,30,11.726029,8.471899
3,Soy_7,soy,7,18.056903,14.093388
4,Wheat_30,wheat,30,12.423013,9.434581
5,Wheat_7,wheat,7,10.711357,8.102552


### Recursive Forecast Function
To simulate real-world scenarios where we need to predict multiple days into the future, we define a recursive forecasting strategy. This function iteratively feeds its own predictions back into the model to forecast out to a specified horizon. Interactive Plotly charts are generated for visual inspection.


In [10]:
@tf.function
def tf_recursive_forecast(model, initial_seq_batch, steps):
    """
    Performs fast, batched recursive forecasting using a trained Keras model.
    Used primarily for downstream evaluation and inference.
    """
    current_seq = tf.cast(initial_seq_batch, tf.float32)
    predictions = tf.TensorArray(tf.float32, size=steps)
    
    for i in tf.range(steps):
        next_pred = model(current_seq, training=False) 
        predictions = predictions.write(i, next_pred)
        
        # Reshape and append the new prediction to the end of the sequence
        next_pred_reshaped = tf.expand_dims(next_pred, axis=1)
        current_seq = tf.concat([current_seq[:, 1:, :], next_pred_reshaped], axis=1)
        
    # Stack the TensorArray and transpose to match (batch_size, steps)
    return tf.transpose(tf.squeeze(predictions.stack(), axis=-1), perm=[1, 0])

def evaluate_recursive_forecast(production_models, df, forecast_horizon_days=30):
    """
    Runs a recursive forecast evaluation for a dictionary of models over a specified horizon.
    Generates interactive Plotly graphs and returns a summary DataFrame of the metrics.
    """
    print(f"🚀 Running {forecast_horizon_days}-Day Recursive Forecast Evaluation...\n")
    
    test_results = []

    for exp_name, model in production_models.items():
        
        # 1. Setup details based on experiment name
        parts = exp_name.split("_")
        commodity = parts[0].lower()
        time_steps = int(parts[1])
        target_col = f"{commodity}_Close" if f"{commodity}_Close" in df.columns else commodity
        
        # Load the specific scaler for this commodity
        scaler_path = f"saved_scalers/{commodity.lower()}/scaler_params.json" 
        try:
            with open(scaler_path, "r") as f:
                scaler_params = json.load(f)
                
                scaler = MinMaxScaler(feature_range=tuple(scaler_params['feature_range']))
                
                scaler.min_ = np.array(scaler_params['min_'])
                scaler.scale_ = np.array(scaler_params['scale_'])
                scaler.data_min_ = np.array(scaler_params['data_min_'])
                scaler.data_max_ = np.array(scaler_params['data_max_'])
                
                scaler.data_range_ = scaler.data_max_ - scaler.data_min_
                
                print(f"   ✅ Scaler loaded successfully for {commodity}")
        except Exception as e:
            print(f"   ❌ Failed to load scaler at {scaler_path}: {e}")
            continue 

        # ==========================================
        # 2. SLICE THE DATASET (End of Timeline)
        # ==========================================
        # Get the raw target series
        target_series = df[target_col].values.reshape(-1, 1)
        
        # Calculate exact indices for the final slice
        seq_start = len(target_series) - forecast_horizon_days - time_steps
        seq_end = len(target_series) - forecast_horizon_days
        
        # Extract the sequence that happened right BEFORE the forecast period
        initial_seq_raw = target_series[seq_start:seq_end]
        
        # Extract the TRUE future we are trying to predict
        actual_future_raw = target_series[seq_end:]
        
        # ==========================================
        # 3. RUN RECURSIVE FORECAST
        # ==========================================
        # Scale the input and format for TensorFlow: (batch=1, time_steps, features=1)
        initial_seq_scaled = scaler.transform(initial_seq_raw)
        initial_seq_tf = tf.convert_to_tensor(initial_seq_scaled.reshape(1, time_steps, 1), dtype=tf.float32)
        
        # Run your custom recursive function
        recursive_preds_scaled = tf_recursive_forecast(model, initial_seq_tf, steps=forecast_horizon_days)
        
        # Inverse transform the results back to real prices
        recursive_preds_real = scaler.inverse_transform(recursive_preds_scaled.numpy().reshape(-1, 1)).flatten()
        actual_future_real = actual_future_raw.flatten()
        
        # ==========================================
        # 4. GET DATES & SLICE FOR PLOTLY
        # ==========================================
        # We use overlap slicing (-1 / +1) strictly so Plotly draws continuous lines.
        # This does NOT affect your RMSE/MAE metrics.
        
        days_of_history = 365
        plot_start_idx = max(0, seq_start - days_of_history)
        
        # 1. Distant Past (Connects perfectly to seq_start)
        past_dates = df['Date'].iloc[plot_start_idx : seq_start + 1]
        past_vals = target_series[plot_start_idx : seq_start + 1].flatten()

        # 2. Input Window (The actual time_steps fed into the model)
        input_dates = df['Date'].iloc[seq_start : seq_end]
        input_vals = target_series[seq_start : seq_end].flatten()

        # 3. Actual Future (Starts 1 day back at 'seq_end - 1' to connect the visual line)
        future_dates = df['Date'].iloc[seq_end - 1 : ] 
        actual_future_vals = target_series[seq_end - 1 : ].flatten()

        # 4. Recursive Forecast (Prepend the last known day so it branches off the black line)
        last_known_val = target_series[seq_end - 1][0]
        recursive_preds_plot = np.insert(recursive_preds_real, 0, last_known_val)

        # ==========================================
        # 5. BUILD INTERACTIVE PLOTLY GRAPH
        # ==========================================
        fig = go.Figure()

        # Trace 1: Distant Past (Light Gray)
        fig.add_trace(go.Scatter(
            x=past_dates, y=past_vals, 
            mode='lines', name='Historical Data (Unused)',
            line=dict(color='lightgray', width=1.5)
        ))

        # Trace 2: The Input Window (Dark Gray/Black)
        fig.add_trace(go.Scatter(
            x=input_dates, y=input_vals, 
            mode='lines', name=f'Input Window ({time_steps} Days)',
            line=dict(color='#333333', width=2.5)
        ))

        # Trace 3: The True Future (Blue)
        fig.add_trace(go.Scatter(
            x=future_dates, y=actual_future_vals, 
            mode='lines', name='Actual Test Data',
            line=dict(color='#1f77b4', width=2.5)
        ))

        # Trace 4: The Recursive Forecast (Red Dashed)
        fig.add_trace(go.Scatter(
            x=future_dates, y=recursive_preds_plot, 
            mode='lines', name=f'{forecast_horizon_days}-Day Forecast',
            line=dict(color='#d62728', width=2.5, dash='dash')
        ))

        # FIXED: Vertical line is now exactly on the last known day of data
        pivot_date = df['Date'].iloc[seq_end - 1]
        fig.add_vline(x=pivot_date, line_width=1.5, line_dash="dot", line_color="black")

        # Layout with your Range Selector
        fig.update_layout(
            title=f"<b>{exp_name}</b>: Full Timeline & Recursive Forecast",
            xaxis_title="Date",
            yaxis_title="Price",
            template="plotly_white",
            hovermode="x unified",
            margin=dict(l=40, r=40, t=60, b=40),
            legend=dict(
                orientation="h", 
                yanchor="bottom", y=1.02, 
                xanchor="right", x=1
            ),
            xaxis=dict(
                rangeselector=dict(
                    buttons=list([
                        dict(count=1, label="1m", step="month", stepmode="backward"),
                        dict(count=6, label="6m", step="month", stepmode="backward"),
                        dict(step="all")
                    ])
                ),
                rangeslider=dict(visible=True),
                type="date"
            )
        )

        fig.show()
        
        # Calculate and print metrics for this recursive slice
        rec_rmse = np.sqrt(mean_squared_error(actual_future_real, recursive_preds_real))
        rec_mae = mean_absolute_error(actual_future_real, recursive_preds_real)
        
        # Store results for the final summary table
        test_results.append({
            "Model": exp_name,
            "Commodity": commodity,
            "Horizon (Days)": forecast_horizon_days,
            "Recursive RMSE": rec_rmse,
            "Recursive MAE": rec_mae
        })

    # Return a clean DataFrame with all the metrics sorted by performance
    return pd.DataFrame(test_results)

### Recursive Forecast Evaluation (7 days in the future)

In [11]:
results_7_days = evaluate_recursive_forecast(production_models, df, forecast_horizon_days=7)
print("\n📊 Final Evaluation Results:")
display(results_7_days)

🚀 Running 7-Day Recursive Forecast Evaluation...

   ✅ Scaler loaded successfully for corn


   ✅ Scaler loaded successfully for corn


   ✅ Scaler loaded successfully for soy


   ✅ Scaler loaded successfully for soy


   ✅ Scaler loaded successfully for wheat


   ✅ Scaler loaded successfully for wheat



📊 Final Evaluation Results:


,Model,Commodity,Horizon (Days),Recursive RMSE,Recursive MAE
0,Corn_30,corn,7,11.266694,11.117985
1,Corn_7,corn,7,7.219020,6.958588
2,Soy_30,soy,7,11.269462,9.383824
3,Soy_7,soy,7,11.908293,10.143659
4,Wheat_30,wheat,7,25.128957,24.629299
5,Wheat_7,wheat,7,19.313611,18.512896


### Recursive Forecast Evaluation (30 days in the future)

In [12]:
results_30_days = evaluate_recursive_forecast(production_models, df, forecast_horizon_days=30)
print("\n📊 Final Evaluation Results:")
display(results_30_days)

🚀 Running 30-Day Recursive Forecast Evaluation...

   ✅ Scaler loaded successfully for corn


   ✅ Scaler loaded successfully for corn


   ✅ Scaler loaded successfully for soy


   ✅ Scaler loaded successfully for soy


   ✅ Scaler loaded successfully for wheat


   ✅ Scaler loaded successfully for wheat



📊 Final Evaluation Results:


,Model,Commodity,Horizon (Days),Recursive RMSE,Recursive MAE
0,Corn_30,corn,30,11.787910,10.031924
1,Corn_7,corn,30,6.723679,5.654125
2,Soy_30,soy,30,62.882327,62.581177
3,Soy_7,soy,30,42.031932,41.176095
4,Wheat_30,wheat,30,17.930570,13.767570
5,Wheat_7,wheat,30,10.621998,8.269407
